# 아이오닉 5 PDF 소제목별 텍스트·이미지 청킹

수업에서 배운 `pypdf → Document → 청킹 → Pandas` 흐름에 이미지 저장과 연결을 추가합니다.

- **텍스트 청크:** 소제목의 설명을 짧게 나눈 글
- **이미지 청크:** 그림 한 장 + 직접 확인한 설명 + 이미지 파일 경로
- 같은 소제목의 텍스트와 이미지는 `parent_id`로 연결합니다.

기존 예제인 PDF 21페이지에는 추출 가능한 이미지가 없습니다. 이번에는 그림이 있는 **PDF 18페이지(인쇄 10쪽)의 ‘비상시 충전 커넥터 분리하기’**를 사용합니다.

이미지를 글자처럼 자르거나 이미지 파일 자체를 텍스트 임베딩에 넣지는 않습니다. 나중에 이미지 설명을 검색하고 연결된 그림을 보여주는 구조입니다. API 호출은 없습니다.


## 1. 도구 불러오기

프로젝트의 `.venv` 커널을 선택한 뒤 위에서부터 실행하세요. 이미지 추출에는 `Pillow`도 필요하며 프로젝트 의존성에 추가했습니다. 다른 환경에서는 프로젝트 루트에서 `uv sync`를 실행하세요.


In [ ]:
from pathlib import Path
import json

from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import display, Image
import pandas as pd


## 2. PDF와 저장 폴더 준비

`data` 폴더는 이 개인 작업 폴더의 `.gitignore`에서 제외되어 있습니다. 이미지와 JSON은 이곳에 저장합니다.


In [ ]:
pdf_path = "/Users/anna/Downloads/NE1_2027_ko_KR.pdf"
work_dir = Path("/Users/anna/mle-02-p1-team1/src/car_search_rag/anna_rag")
output_dir = work_dir / "data" / "manual_chunks" / Path(pdf_path).stem
output_dir.mkdir(parents=True, exist_ok=True)

reader = PdfReader(pdf_path)
print("전체 페이지 수:", len(reader.pages))
print("저장 위치:", output_dir)


## 3. 소제목과 페이지 선택

이 예제는 한 페이지 안에서 끝나는 소제목을 직접 선택합니다. 다른 페이지를 사용하면 제목·쪽수·ID·이미지 설명도 함께 수정해야 합니다.

기존 텍스트 예제로 돌아가려면 `target_page=21`, `printed_page="13"`, `title="급속 충전기로 충전하기"`, `next_title="휴대용 충전기(ICCB) 사용하기"`, `parent_id="fast_charge_p21"`로 바꾸고 6번의 이미지 선택을 빈 딕셔너리 `{}`로 설정하세요.


In [ ]:
target_page = 18
printed_page = "10"
title = "비상시 충전 커넥터 분리하기"
next_title = "완속 충전기 사용하기"
parent_id = "emergency_connector_p18"

page = reader.pages[target_page - 1]
page_text = page.extract_text() or ""
print(page_text)


## 4. 소제목의 텍스트 선택·정리

시작 제목부터 다음 제목 직전까지 선택합니다. 다음 제목이 다른 페이지에 있는 경우에는 먼저 페이지를 연결하는 별도 처리가 필요합니다.

세 가지 깨진 표기는 이 PDF의 원본과 비교한 값입니다. 모든 PDF에 적용되는 규칙은 아닙니다.


In [ ]:
start = page_text.index(title)
end = page_text.index(next_title, start + len(title))
raw_section_text = page_text[start:end]
section_text = raw_section_text.strip()

section_text = section_text.replace("҃Ҋ", "[경고]")
section_text = section_text.replace("઱੄", "[주의]")
section_text = section_text.replace("ӝ", "[알아두기]")
print(section_text)


## 5. 페이지의 이미지 미리보기

이미지 번호는 PDF의 저장 순서이며, 위에서 아래로 보이는 순서와 다를 수 있습니다. 원본 페이지와 미리보기를 비교해 해당 소제목의 그림만 고르세요.

`page.images`는 PDF에 삽입된 이미지 객체를 꺼냅니다. 벡터로 그려진 그림·아이콘은 빠질 수 있고, 이미지 속 글자를 읽는 OCR은 수행하지 않습니다. 이번 페이지에서는 큰 설명 그림 1장을 추출하며 경고·주의 아이콘은 포함하지 않습니다.


In [ ]:
page_images = list(page.images)
print("추출 가능한 이미지 수:", len(page_images))

for image_no, pdf_image in enumerate(page_images, start=1):
    print("이미지 번호:", image_no, "원본 이름:", pdf_image.name)
    # 이 PDF의 CMYK 그림도 브라우저에서 보이도록 RGB로 변환합니다.
    preview = pdf_image.image.convert("RGB")
    display(preview)

if not page_images:
    print("삽입 이미지가 없습니다. 벡터 그림 유무는 원본 PDF에서 확인하세요.")


## 6. 소제목에 해당하는 이미지 선택

번호를 직접 선택하는 이유는 같은 페이지에 다른 소제목의 이미지가 있을 수 있기 때문입니다. 이미지 설명은 원본과 주변 본문을 보고 직접 작성합니다. AI가 그림을 해석한 결과가 아닙니다.

18페이지의 1번 그림을 확인하여 아래에 연결했습니다. 여러 그림이면 `2: "두 번째 그림 설명"`처럼 추가합니다. 이미지가 없는 소제목이면 `{}`로 둡니다.


In [ ]:
selected_image_captions = {
    1: "화물칸의 충전 커넥터 비상 해제 케이블을 확대해 보여주는 그림"
}

# 잘못된 번호 또는 빈 설명이면 파일을 저장하기 전에 알려줍니다.
for image_no, caption in selected_image_captions.items():
    if image_no < 1 or image_no > len(page_images):
        raise ValueError(f"이미지 {image_no}번이 없습니다. 5번 미리보기를 확인하세요.")
    if not caption.strip():
        raise ValueError("검색에 사용할 이미지 설명을 입력하세요.")


## 7. 이미지 파일과 이미지 청크 만들기

원본 파일과 화면 표시용 PNG를 함께 저장합니다. PNG에는 해당 이미지 안의 글자·선이 보존되지만, 이미지 바깥에 따로 그려진 표식은 포함되지 않을 수 있습니다.

`page_content`에는 검색용 제목·설명을 넣고, `metadata`에는 그림 경로와 출처를 넣습니다. `image_path`는 `output_dir` 기준 상대 경로입니다.


In [ ]:
source_id = Path(pdf_path).stem
section_dir = output_dir / parent_id
section_dir.mkdir(parents=True, exist_ok=True)

common_metadata = {
    "source": Path(pdf_path).name,
    "source_id": source_id,
    "vehicle_id": "ioniq5",
    "pdf_page": target_page,
    "printed_page": printed_page,
    "title": title,
    "parent_id": parent_id
}

image_chunks = []
for image_no, caption in selected_image_captions.items():
    pdf_image = page_images[image_no - 1]
    image_id = f"{source_id}_{parent_id}_image_{image_no:02d}"

    # PDF 속 이미지의 원본 데이터를 저장합니다.
    original_suffix = Path(pdf_image.name).suffix
    original_path = section_dir / f"{image_id}_original{original_suffix}"
    original_path.write_bytes(pdf_image.data)

    # 화면 표시용 PNG는 RGB로 변환하여 저장합니다.
    image_path = section_dir / f"{image_id}.png"
    pdf_image.image.convert("RGB").save(image_path)

    metadata = common_metadata.copy()
    metadata.update({
        "chunk_id": image_id,
        "chunk_type": "image",
        "image_no": image_no,
        "image_path": image_path.relative_to(output_dir).as_posix(),
        "original_image_path": original_path.relative_to(output_dir).as_posix(),
        "caption_source": "manual"
    })
    image_doc = Document(
        page_content=f"소제목: {title}\n이미지 설명: {caption}",
        metadata=metadata
    )
    image_chunks.append(image_doc)

print("이미지 청크 수:", len(image_chunks))


## 8. 전체 설명 Document 만들기

`parent_doc`에는 경고·주의를 포함한 전체 설명을 보관합니다. 이미지 청크나 작은 텍스트 청크가 검색되면 `parent_id`로 이 설명을 함께 가져옵니다.


In [ ]:
parent_doc = Document(
    page_content=section_text,
    metadata=common_metadata.copy()
)
pdf_docs = [parent_doc]
print(parent_doc.metadata)


## 9. 텍스트 청킹 후 이미지 청크와 합치기

텍스트만 `split_documents()`로 나눕니다. 이미지는 한 장이 한 청크이므로 분할하지 않습니다. `400/80`은 수업에서 사용한 출발값이며, 이미지 해상도와 관계없는 글자 수 설정입니다.


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)
chunks = splitter.split_documents(pdf_docs)

for i, chunk in enumerate(chunks, start=1):
    chunk.metadata["chunk_no"] = i
    chunk.metadata["chunk_type"] = "text"
    chunk.metadata["chunk_id"] = f"{source_id}_{parent_id}_text_{i:02d}"

all_chunks = chunks + image_chunks
print("텍스트 청크:", len(chunks))
print("이미지 청크:", len(image_chunks))
print("전체 청크:", len(all_chunks))


## 10. Pandas로 결과 확인

한 행이 텍스트 또는 이미지 청크 하나입니다. 텍스트 행에는 이미지 경로가 없어 빈 문자열을 넣습니다.


In [ ]:
rows = []
for chunk in all_chunks:
    rows.append({
        "청크ID": chunk.metadata["chunk_id"],
        "종류": chunk.metadata["chunk_type"],
        "소제목": chunk.metadata["title"],
        "PDF페이지": chunk.metadata["pdf_page"],
        "전체설명ID": chunk.metadata["parent_id"],
        "내용": chunk.page_content,
        "이미지경로": chunk.metadata.get("image_path", "")
    })

chunks_df = pd.DataFrame(rows)
print("행·열 수:", chunks_df.shape)
print(chunks_df.dtypes)
print("결측치 개수:")
print(chunks_df.isna().sum())
display(chunks_df)


## 11. 검색된 청크의 전체 설명과 관련 그림 가져오기

아직 검색기를 만들지 않았으므로 첫 텍스트 청크가 검색되었다고 가정합니다. `selected_chunk = image_chunks[0]`으로 바꾸면 이미지 청크에서 전체 설명을 찾는 흐름도 확인할 수 있습니다.

연결된 그림은 **같은 소제목 단위의 참고 자료**입니다. 각각의 짧은 텍스트 청크와 그림이 일대일로 대응한다는 뜻은 아닙니다.


In [ ]:
selected_chunk = all_chunks[0]
selected_parent_id = selected_chunk.metadata["parent_id"]
selected_source_id = selected_chunk.metadata["source_id"]

for doc in pdf_docs:
    if (doc.metadata["parent_id"] == selected_parent_id
            and doc.metadata["source_id"] == selected_source_id):
        print(doc.page_content)
        print("출처:", doc.metadata["source"], "PDF", doc.metadata["pdf_page"], "페이지")

for image_doc in image_chunks:
    if (image_doc.metadata["parent_id"] == selected_parent_id
            and image_doc.metadata["source_id"] == selected_source_id):
        print(image_doc.page_content)
        display(Image(filename=str(output_dir / image_doc.metadata["image_path"])))


## 12. 다음 실습에서 사용할 JSON 저장

전체 설명, 텍스트 청크, 이미지 청크를 저장합니다. 이미지 파일은 별도로 보관하므로 다른 컴퓨터로 옮길 때는 `output_dir` 폴더 전체를 복사하세요. 같은 소제목을 다시 실행하면 같은 JSON 파일을 갱신합니다. 다른 소제목은 다른 `parent_id`를 사용하세요.

이미지 선택을 바꾸면 이전 이미지 파일이 폴더에 남을 수 있습니다. 다음 단계에서는 폴더 전체를 적재하지 말고, 이 JSON에 등록된 청크만 읽으세요.


In [ ]:
records = []
for chunk in all_chunks:
    records.append({
        "page_content": chunk.page_content,
        "metadata": chunk.metadata
    })

result = {
    "parent": {
        "page_content": parent_doc.page_content,
        "metadata": parent_doc.metadata
    },
    "chunks": records
}
json_path = output_dir / f"{parent_id}.json"
json_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장 완료:", json_path)

# 저장한 결과를 다시 읽어 청크 수를 확인합니다.
saved = json.loads(json_path.read_text(encoding="utf-8"))
print("저장된 청크 수:", len(saved["chunks"]))


## 이번 단계의 결과

- `pdf_docs`: 경고·주의를 포함한 소제목 전체 설명
- `chunks`: 작은 텍스트 청크
- `image_chunks`: 그림의 설명·파일 경로를 가진 이미지 청크
- `all_chunks`, `chunks_df`: 텍스트와 이미지 청크를 합친 결과
- `data/manual_chunks/...`: 원본 이미지, 표시용 PNG, 청크 JSON

**다음 단계:** `all_chunks`의 `page_content`를 임베딩하고 메타데이터와 함께 DB에 저장합니다. 검색 결과의 `parent_id`로 전체 설명과 그림을 연결합니다.

이 노트북은 한 소제목의 실습입니다. 전체 PDF 자동 소제목 분류, 이미지 자동 해석, OCR, 벡터 그림 추출, 임베딩·DB 저장은 아직 수행하지 않습니다.

참고: [pypdf 공식 이미지 추출 문서](https://pypdf.readthedocs.io/en/stable/user/extract-images.html)
